In [ ]:
! pip install transformers datasets
pip install evaluate

In [ ]:
def compute_metrics(eval_preds):
    predictions, references = eval_preds
    correct = 0
    total = len(predictions)

    for pred_ids, label_ids in zip(predictions, references):
        # Remove ignored tokens
        label_ids = label_ids[label_ids != -100]
        pred_ids = pred_ids[:len(label_ids)]

        # Decode tokens to string
        pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True).strip()
        label_text = tokenizer.decode(label_ids, skip_special_tokens=True).strip()

        if pred_text == label_text:
            correct += 1

    return {"exact_match": correct / total}


In [ ]:
import torch
from transformers import PreTrainedTokenizerFast, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from transformers import DeepseekV3ForCausalLM  

# -------------------------------
# Paths & Hyperparameters
# -------------------------------
model_path = "/kaggle/input/data-pretrained-model"
tokenizer_path = "/kaggle/input/tokeniser/multilingual_bpe_64k.json"

train_file = "/kaggle/input/syllable-split/syllable_train.jsonl"
val_file   = "/kaggle/input/syllable-split/syllable_val.jsonl"
test_file  = "/kaggle/input/syllable-split/syllable_test.jsonl"

max_length = 96
response_max_length = 32

# -------------------------------
# Load tokenizer
# -------------------------------
tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
tokenizer.eos_token = "</s>"
tokenizer.pad_token = "<pad>"
tokenizer.padding_side = "left"

# -------------------------------
# Load pretrained Deepseek model
# -------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DeepseekV3ForCausalLM.from_pretrained(model_path, trust_remote_code=True)
model.to(device)
model.train()

# -------------------------------
# Load dataset
# -------------------------------
dataset = load_dataset("json", data_files={
    "train": train_file,
    "validation": val_file,
    "test": test_file
})

# -------------------------------
# Preprocess dataset
# -------------------------------
def preprocess(example):
    prompt = f"User: {example['prompt']}\nAssistant:"
    target = example['response']
    full_text = prompt + " " + target + tokenizer.eos_token

    tokenized = tokenizer(
        full_text,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    # Mask prompt tokens in labels
    prompt_len = len(tokenizer(prompt)["input_ids"])
    labels = tokenized["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len  # use list of -100
    tokenized["labels"] = labels

    return tokenized


tokenized_dataset = dataset.map(preprocess, batched=False)

# -------------------------------
# Data collator
# -------------------------------
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# -------------------------------
# Training arguments
# -------------------------------
training_args = TrainingArguments(
    output_dir="./reverse_finetune",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    learning_rate=1e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=True,
    report_to="none"
)

# -------------------------------
# Trainer
# -------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# -------------------------------
# Train
# -------------------------------
trainer.train()


In [ ]:
eval_results = trainer.evaluate(eval_dataset=tokenized_dataset["validation"])
print("Validation Perplexity:", torch.exp(torch.tensor(eval_results["eval_loss"])))


In [ ]:
model.eval()
correct = 0
total = 0

for example in dataset["test"]:
    prompt = f"User: Reverse the word: {example['prompt']}\nAssistant:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    if "token_type_ids" in inputs:  # remove for decoder-only model
        inputs.pop("token_type_ids")

    outputs = model.generate(
        **inputs,
        max_new_tokens=response_max_length,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Only take generated tokens after the prompt
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]
    pred_word = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    if pred_word == example["response"]:
        correct += 1
    total += 1

print("Test Exact-match Accuracy:", correct / total)


In [ ]:
# Directory where you want to save
save_dir = "./finetuned_model_syllable_task"

# Save the model
model.save_pretrained(save_dir)

# Save the tokenizer
tokenizer.save_pretrained(save_dir)
